# Variante B: previsão rolling de um passo

Na variante B cada previsão é de um passo, e o valor real chega logo depois. Os parâmetros ficam congelados:

1. prever `t+1` usando só o que foi observado até `t`;
2. observar `y[t+1]`;
3. atualizar a janela de resíduos com o erro da base naquele instante;
4. avançar, **sem chamar `fit`**.

A seleção usa o erro de um passo na validação. A seleção dinâmica de corretores só é permitida nesta variante.

In [ ]:
import autohybts

print("AutoHybTS", autohybts.__version__)

## 1. Experimento na variante B

In [ ]:
from autohybts import AutoHybridExperiment
from autohybts.datasets import get_data

y = get_data("synthetic_daily")
exp = AutoHybridExperiment(fh=1, variant="B", test_size=28, session_id=42).fit(y)
result = exp.compare_models()
result.leaderboard

## 2. Finalizar e avaliar no teste selado

In [ ]:
final = exp.finalize_model(result.best)
exp.evaluate_external(final).metrics

## 3. Streaming em produção

O mesmo modelo, recebendo observações novas uma a uma. Aqui simulamos a chegada dos dados com as últimas 28 observações da série.

In [ ]:
import pandas as pd

stream = final.stream()
rows = []
for timestamp, actual in y.iloc[-28:].items():
    ticket = stream.predict_next()
    rows.append({"timestamp": timestamp, "y_pred": ticket.prediction, "y_true": actual})
    stream.observe(actual)  # só agora o valor real fica disponível

stream_log = pd.DataFrame(rows).set_index("timestamp")
stream_log["abs_error"] = (stream_log["y_true"] - stream_log["y_pred"]).abs()
stream_log